## 1. Install packages

In [1]:
!pip install -q ultralytics opencv-python pandas matplotlib

## 2. Imports and GPU check

In [2]:
from pathlib import Path
from collections import defaultdict, deque
from datetime import datetime

import cv2
import numpy as np
import pandas as pd
import torch

from ultralytics import YOLO
from google.colab import drive
from IPython.display import Video, display

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
PyTorch: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4


## 3. Mount Google Drive

In [3]:
drive.mount('/content/drive')

Mounted at /content/drive


## 4. Project paths

In [4]:
BASE_DIR = Path("/content/drive/MyDrive/cheating tracking")
VIDEO_DIR = BASE_DIR / "videos"

OUTPUT_DIR = BASE_DIR / "outputs"
FINAL_DIR = OUTPUT_DIR / "final_cheating"
SCREENSHOT_DIR = FINAL_DIR / "screenshots"
ALERT_DIR = FINAL_DIR / "alerts"

TRACKER_CONFIG = BASE_DIR / "tracker_configs" / "bytetrack_exam.yaml"

FINAL_DIR.mkdir(parents=True, exist_ok=True)
SCREENSHOT_DIR.mkdir(parents=True, exist_ok=True)
ALERT_DIR.mkdir(parents=True, exist_ok=True)

print("VIDEO_DIR:", VIDEO_DIR)
print("FINAL_DIR:", FINAL_DIR)
print("SCREENSHOT_DIR:", SCREENSHOT_DIR)
print("ALERT_DIR:", ALERT_DIR)
print("TRACKER_CONFIG:", TRACKER_CONFIG)


VIDEO_DIR: /content/drive/MyDrive/cheating tracking/videos
FINAL_DIR: /content/drive/MyDrive/cheating tracking/outputs/final_cheating
SCREENSHOT_DIR: /content/drive/MyDrive/cheating tracking/outputs/final_cheating/screenshots
ALERT_DIR: /content/drive/MyDrive/cheating tracking/outputs/final_cheating/alerts
TRACKER_CONFIG: /content/drive/MyDrive/cheating tracking/tracker_configs/bytetrack_exam.yaml


## 5. Load videos

In [5]:
video_files = sorted(
    list(VIDEO_DIR.glob("*.mp4")) +
    list(VIDEO_DIR.glob("*.avi")) +
    list(VIDEO_DIR.glob("*.mov")) +
    list(VIDEO_DIR.glob("*.mkv"))
)

for i, video in enumerate(video_files):
    print(i, "->", video.name)

assert len(video_files) > 0, "No videos found."


0 -> video1.mp4
1 -> video2.mp4
2 -> video3.mp4
3 -> video4.mp4


## 6. Global configuration

In [6]:
DEVICE = 0 if torch.cuda.is_available() else "cpu"

PERSON_CLASS = 0
CELL_PHONE_CLASS = 67

# Detection thresholds
PERSON_CONF = 0.30
PHONE_CONF = 0.45
POSE_CONF = 0.30

# Pose is expensive; run once every N frames
POSE_STRIDE = 3

# Final decision thresholds
SIDE_LOOK_THRESHOLD_SEC = 2.5
HEAD_TURN_THRESHOLD = 3
HEAD_TURN_WINDOW_SEC = 8.0

# Confirm phone use temporally
PHONE_CONFIRM_SECONDS = 0.5

# Avoid saving many screenshots for the same student/event
ALERT_COOLDOWN_SECONDS = 5.0

print("Device:", DEVICE)


Device: 0


## 7. Geometry helpers

In [7]:
def box_center(box):
    x1, y1, x2, y2 = box
    return ((x1 + x2) / 2, (y1 + y2) / 2)


def expand_box(box, frame_shape, scale_x=0.25, scale_y=0.15):
    frame_h, frame_w = frame_shape[:2]
    x1, y1, x2, y2 = map(float, box)

    bw = x2 - x1
    bh = y2 - y1

    return (
        max(0, x1 - bw * scale_x),
        max(0, y1 - bh * scale_y),
        min(frame_w - 1, x2 + bw * scale_x),
        min(frame_h - 1, y2 + bh * scale_y),
    )


def point_inside_box(point, box):
    x, y = point
    x1, y1, x2, y2 = box
    return x1 <= x <= x2 and y1 <= y <= y2


def box_iou(box_a, box_b):
    ax1, ay1, ax2, ay2 = map(float, box_a)
    bx1, by1, bx2, by2 = map(float, box_b)

    x1 = max(ax1, bx1)
    y1 = max(ay1, by1)
    x2 = min(ax2, bx2)
    y2 = min(ay2, by2)

    inter_w = max(0, x2 - x1)
    inter_h = max(0, y2 - y1)
    intersection = inter_w * inter_h

    area_a = max(0, ax2 - ax1) * max(0, ay2 - ay1)
    area_b = max(0, bx2 - bx1) * max(0, by2 - by1)

    union = area_a + area_b - intersection

    return intersection / union if union > 0 else 0


## 8. Head direction estimation

In [8]:
NOSE = 0
LEFT_EYE = 1
RIGHT_EYE = 2
LEFT_EAR = 3
RIGHT_EAR = 4


def estimate_head_direction(keypoints_xy, keypoints_conf=None):
    xy = np.asarray(keypoints_xy, dtype=float)

    if xy.ndim != 2 or xy.shape[0] < 5:
        return "unknown"

    if keypoints_conf is None:
        conf = np.ones(xy.shape[0], dtype=float)
    else:
        conf = np.asarray(keypoints_conf, dtype=float)

    def visible(index, threshold=0.25):
        return (
            index < len(conf)
            and conf[index] >= threshold
            and xy[index, 0] > 0
            and xy[index, 1] > 0
        )

    if visible(NOSE) and visible(LEFT_EYE) and visible(RIGHT_EYE):
        nose_x = xy[NOSE, 0]
        left_eye_x = xy[LEFT_EYE, 0]
        right_eye_x = xy[RIGHT_EYE, 0]

        eye_mid = (left_eye_x + right_eye_x) / 2
        eye_distance = abs(left_eye_x - right_eye_x)

        if eye_distance >= 3:
            offset = (nose_x - eye_mid) / eye_distance

            if offset > 0.28:
                return "right"
            elif offset < -0.28:
                return "left"
            else:
                return "forward"

    left_side_visible = visible(LEFT_EYE) or visible(LEFT_EAR)
    right_side_visible = visible(RIGHT_EYE) or visible(RIGHT_EAR)

    if left_side_visible and not right_side_visible:
        return "right"

    if right_side_visible and not left_side_visible:
        return "left"

    return "unknown"


## 9. Match pose result with tracked students

In [9]:
def match_pose_to_tracks(
    tracked_students,
    pose_boxes,
    pose_keypoints_xy,
    pose_keypoints_conf
):
    directions = {}

    for pose_index, pose_box in enumerate(pose_boxes):
        best_track_id = None
        best_iou = 0

        for student in tracked_students:
            score = box_iou(student["box"], pose_box)

            if score > best_iou:
                best_iou = score
                best_track_id = student["track_id"]

        if best_track_id is not None and best_iou >= 0.20:
            kp_conf = None

            if (
                pose_keypoints_conf is not None
                and pose_index < len(pose_keypoints_conf)
            ):
                kp_conf = pose_keypoints_conf[pose_index]

            directions[best_track_id] = estimate_head_direction(
                pose_keypoints_xy[pose_index],
                kp_conf
            )

    return directions


## 10. Associate phones with students

In [10]:
def phone_near_student(student_box, phone_detections, frame_shape):
    search_box = expand_box(
        student_box,
        frame_shape,
        scale_x=0.25,
        scale_y=0.15
    )

    best_phone = None
    best_conf = 0.0

    for phone in phone_detections:
        center = box_center(phone["box"])

        if point_inside_box(center, search_box):
            if phone["confidence"] >= PHONE_CONF and phone["confidence"] > best_conf:
                best_conf = phone["confidence"]
                best_phone = phone

    return best_phone


## 11. Temporal state for every student

We store:
- current head direction
- sideways-looking start time
- recent head direction changes
- phone-confirmation timer
- last alert time


In [11]:
class StudentState:
    def __init__(self):
        self.last_direction = "unknown"
        self.side_started_at = None
        self.direction_changes = deque()

        self.phone_started_at = None
        self.phone_frames = 0

        self.last_alert_at = -10_000.0


student_states = defaultdict(StudentState)


## 12. Update temporal behavior

In [12]:
def update_temporal_state(
    track_id,
    direction,
    phone_detected,
    video_time
):
    state = student_states[track_id]

    # ------------------
    # Head direction
    # ------------------

    if direction in {"left", "right"}:

        if (
            state.last_direction in {"left", "right"}
            and direction != state.last_direction
        ):
            state.direction_changes.append(video_time)

        while (
            state.direction_changes
            and video_time - state.direction_changes[0] > HEAD_TURN_WINDOW_SEC
        ):
            state.direction_changes.popleft()

        if state.side_started_at is None:
            state.side_started_at = video_time

        side_duration = video_time - state.side_started_at

    else:
        state.side_started_at = None
        side_duration = 0.0

    if direction != "unknown":
        state.last_direction = direction

    # ------------------
    # Phone confirmation
    # ------------------

    if phone_detected:
        state.phone_frames += 1

        if state.phone_started_at is None:
            state.phone_started_at = video_time

        phone_duration = video_time - state.phone_started_at

    else:
        state.phone_frames = 0
        state.phone_started_at = None
        phone_duration = 0.0

    return {
        "side_look_duration": side_duration,
        "head_turn_count": len(state.direction_changes),
        "phone_duration": phone_duration,
        "phone_frames": state.phone_frames,
    }


## 13. Final cheating decision logic

The decision uses **temporal confirmation**.

### Rule A — Confirmed phone
Phone must remain associated with the student for at least:

```text
0.5 seconds
```

### Rule B — Prolonged sideways looking

```text
>= 2.5 seconds
```

### Rule C — Repeated head turns

```text
>= 3 turns within 8 seconds
```


In [13]:
def make_cheating_decision(
    phone_detected,
    phone_confidence,
    temporal
):
    confirmed_phone = (
        phone_detected
        and temporal["phone_duration"] >= PHONE_CONFIRM_SECONDS
    )

    prolonged_side_look = (
        temporal["side_look_duration"] >= SIDE_LOOK_THRESHOLD_SEC
    )

    repeated_head_turns = (
        temporal["head_turn_count"] >= HEAD_TURN_THRESHOLD
    )

    if confirmed_phone:
        return {
            "is_cheating": True,
            "behavior": "Phone detected",
            "confidence": max(0.80, float(phone_confidence)),
        }

    if prolonged_side_look:
        # Rule-based score, not model probability
        score = min(
            0.95,
            0.70 + 0.05 * (
                temporal["side_look_duration"]
                - SIDE_LOOK_THRESHOLD_SEC
            )
        )

        return {
            "is_cheating": True,
            "behavior": "Prolonged sideways looking",
            "confidence": score,
        }

    if repeated_head_turns:
        score = min(
            0.95,
            0.75 + 0.05 * (
                temporal["head_turn_count"]
                - HEAD_TURN_THRESHOLD
            )
        )

        return {
            "is_cheating": True,
            "behavior": "Repeated head turns",
            "confidence": score,
        }

    return {
        "is_cheating": False,
        "behavior": "Normal",
        "confidence": 0.0,
    }


## 14. Alert cooldown logic

In [14]:
def can_create_alert(track_id, video_time):
    state = student_states[track_id]

    return (
        video_time - state.last_alert_at
        >= ALERT_COOLDOWN_SECONDS
    )


def mark_alert_created(track_id, video_time):
    student_states[track_id].last_alert_at = video_time


## 15. Screenshot saving helper

In [15]:
def save_alert_screenshot(
    frame,
    video_name,
    track_id,
    behavior,
    video_time
):
    safe_behavior = (
        behavior.lower()
        .replace(" ", "_")
        .replace("/", "_")
    )

    timestamp = datetime.now().strftime(
        "%Y%m%d_%H%M%S_%f"
    )

    filename = (
        f"{Path(video_name).stem}"
        f"_student_{track_id}"
        f"_{safe_behavior}"
        f"_{timestamp}.jpg"
    )

    screenshot_path = (
        SCREENSHOT_DIR / filename
    )

    cv2.imwrite(
        str(screenshot_path),
        frame
    )

    return screenshot_path


## 16. Final integrated processing function

This function performs:

```text
YOLO11
  ↓
Person + Phone Detection
  ↓
ByteTrack
  ↓
Pose
  ↓
Behavior Signals
  ↓
Cheating Decision
  ↓
Green / Red Bounding Box
  ↓
Screenshot
  ↓
Alert History Record
```


In [16]:
def process_final_cheating_video(
    input_video,
    output_video,
    alert_csv
):
    global student_states

    student_states = defaultdict(StudentState)

    tracker_model = YOLO("yolo11n.pt")
    pose_model = YOLO("yolo11n-pose.pt")

    cap = cv2.VideoCapture(str(input_video))

    if not cap.isOpened():
        raise RuntimeError(
            f"Cannot open video: {input_video}"
        )

    fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(
        cap.get(cv2.CAP_PROP_FRAME_COUNT)
    )

    width = int(
        cap.get(cv2.CAP_PROP_FRAME_WIDTH)
    )

    height = int(
        cap.get(cv2.CAP_PROP_FRAME_HEIGHT)
    )

    fourcc = cv2.VideoWriter_fourcc(
        *"mp4v"
    )

    writer = cv2.VideoWriter(
        str(output_video),
        fourcc,
        fps,
        (width, height)
    )

    frame_index = 0

    last_head_direction = {}

    alert_records = []

    while True:

        ok, frame = cap.read()

        if not ok:
            break

        video_time = (
            frame_index / fps
            if fps > 0
            else 0
        )

        # -------------------------------------
        # Detection + Tracking
        # -------------------------------------

        result = tracker_model.track(
            source=frame,
            persist=True,
            tracker=str(TRACKER_CONFIG),
            classes=[
                PERSON_CLASS,
                CELL_PHONE_CLASS
            ],
            conf=PERSON_CONF,
            device=DEVICE,
            verbose=False
        )[0]

        students = []
        phones = []

        if (
            result.boxes is not None
            and len(result.boxes) > 0
        ):

            boxes = (
                result.boxes.xyxy
                .cpu()
                .numpy()
            )

            classes = (
                result.boxes.cls
                .int()
                .cpu()
                .tolist()
            )

            confidences = (
                result.boxes.conf
                .cpu()
                .tolist()
            )

            if result.boxes.id is not None:
                ids = (
                    result.boxes.id
                    .int()
                    .cpu()
                    .tolist()
                )
            else:
                ids = [-1] * len(boxes)

            for box, cls_id, conf, tid in zip(
                boxes,
                classes,
                confidences,
                ids
            ):

                item = {
                    "box": tuple(
                        map(float, box)
                    ),
                    "confidence": float(conf),
                    "track_id": int(tid)
                }

                if (
                    cls_id == PERSON_CLASS
                    and tid >= 0
                ):
                    students.append(item)

                elif cls_id == CELL_PHONE_CLASS:
                    phones.append(item)

        # -------------------------------------
        # Pose estimation
        # -------------------------------------

        if (
            frame_index % max(1, POSE_STRIDE) == 0
            and len(students) > 0
        ):

            pose_result = pose_model.predict(
                source=frame,
                conf=POSE_CONF,
                device=DEVICE,
                verbose=False
            )[0]

            if (
                pose_result.boxes is not None
                and pose_result.keypoints is not None
            ):

                pose_boxes = (
                    pose_result.boxes.xyxy
                    .cpu()
                    .numpy()
                )

                kp_xy = (
                    pose_result.keypoints.xy
                    .cpu()
                    .numpy()
                )

                kp_conf = None

                if pose_result.keypoints.conf is not None:
                    kp_conf = (
                        pose_result.keypoints.conf
                        .cpu()
                        .numpy()
                    )

                directions = match_pose_to_tracks(
                    students,
                    pose_boxes,
                    kp_xy,
                    kp_conf
                )

                last_head_direction.update(
                    directions
                )

        # -------------------------------------
        # Analyze students before drawing
        # -------------------------------------

        frame_student_results = []

        for student in students:

            tid = student["track_id"]

            direction = last_head_direction.get(
                tid,
                "unknown"
            )

            phone = phone_near_student(
                student["box"],
                phones,
                frame.shape
            )

            phone_detected = (
                phone is not None
            )

            phone_confidence = (
                float(phone["confidence"])
                if phone is not None
                else 0.0
            )

            temporal = update_temporal_state(
                tid,
                direction,
                phone_detected,
                video_time
            )

            decision = make_cheating_decision(
                phone_detected,
                phone_confidence,
                temporal
            )

            frame_student_results.append({
                "student": student,
                "track_id": tid,
                "direction": direction,
                "phone_detected": phone_detected,
                "phone_confidence": phone_confidence,
                "temporal": temporal,
                "decision": decision
            })

        # -------------------------------------
        # Draw annotations
        # -------------------------------------

        annotated = frame.copy()

        # Draw phones orange
        for phone in phones:

            if phone["confidence"] < PHONE_CONF:
                continue

            px1, py1, px2, py2 = map(
                int,
                phone["box"]
            )

            cv2.rectangle(
                annotated,
                (px1, py1),
                (px2, py2),
                (0, 165, 255),
                2
            )

            cv2.putText(
                annotated,
                f"Phone {phone['confidence']:.2f}",
                (px1, max(20, py1 - 8)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.55,
                (0, 165, 255),
                2
            )

        # Draw student boxes
        for item in frame_student_results:

            student = item["student"]
            tid = item["track_id"]
            direction = item["direction"]
            temporal = item["temporal"]
            decision = item["decision"]

            x1, y1, x2, y2 = map(
                int,
                student["box"]
            )

            if decision["is_cheating"]:
                color = (0, 0, 255)
                status = "SUSPECTED CHEATING"
            else:
                color = (0, 255, 0)
                status = "NORMAL"

            cv2.rectangle(
                annotated,
                (x1, y1),
                (x2, y2),
                color,
                3
            )

            label1 = (
                f"Student {tid} | {status}"
            )

            label2 = (
                f"{decision['behavior']} | "
                f"head:{direction}"
            )

            cv2.putText(
                annotated,
                label1,
                (x1, max(20, y1 - 28)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.60,
                color,
                2
            )

            cv2.putText(
                annotated,
                label2,
                (x1, max(20, y1 - 7)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.50,
                color,
                2
            )

        # -------------------------------------
        # Save alert screenshots AFTER red box
        # is already drawn
        # -------------------------------------

        for item in frame_student_results:

            tid = item["track_id"]
            decision = item["decision"]
            temporal = item["temporal"]

            if (
                decision["is_cheating"]
                and can_create_alert(
                    tid,
                    video_time
                )
            ):

                screenshot_path = save_alert_screenshot(
                    annotated,
                    input_video.name,
                    tid,
                    decision["behavior"],
                    video_time
                )

                event_time = datetime.now()

                alert_record = {
                    "video": input_video.name,
                    "student_track_id": tid,
                    "behavior_detected": decision["behavior"],
                    "confidence_score": round(
                        float(
                            decision["confidence"]
                        ),
                        4
                    ),
                    "video_time_seconds": round(
                        float(video_time),
                        3
                    ),
                    "event_datetime": event_time.strftime(
                        "%Y-%m-%d %H:%M:%S"
                    ),
                    "screenshot_path": str(
                        screenshot_path
                    ),
                    "head_direction": item[
                        "direction"
                    ],
                    "side_look_duration": round(
                        float(
                            temporal[
                                "side_look_duration"
                            ]
                        ),
                        3
                    ),
                    "head_turn_count": int(
                        temporal[
                            "head_turn_count"
                        ]
                    ),
                    "phone_duration": round(
                        float(
                            temporal[
                                "phone_duration"
                            ]
                        ),
                        3
                    ),
                }

                alert_records.append(
                    alert_record
                )

                mark_alert_created(
                    tid,
                    video_time
                )

                print(
                    "ALERT |",
                    input_video.name,
                    "| Student",
                    tid,
                    "|",
                    decision["behavior"],
                    "|",
                    f"{video_time:.2f}s"
                )

        writer.write(
            annotated
        )

        frame_index += 1

        if frame_index % 50 == 0:
            print(
                f"{input_video.name}: "
                f"{frame_index}/{total_frames}"
            )

    cap.release()
    writer.release()

    alerts_df = pd.DataFrame(
        alert_records
    )

    alerts_df.to_csv(
        alert_csv,
        index=False
    )

    print()
    print("Finished:", input_video.name)
    print("Output video:", output_video)
    print("Alert CSV:", alert_csv)
    print("Number of alerts:", len(alerts_df))

    return alerts_df


## 17. Run the final pipeline on one video first

In [17]:
TEST_VIDEO = video_files[0]

OUTPUT_VIDEO = (
    FINAL_DIR /
    f"{TEST_VIDEO.stem}_final_cheating.mp4"
)

ALERT_CSV = (
    ALERT_DIR /
    f"{TEST_VIDEO.stem}_alerts.csv"
)

alerts_df = process_final_cheating_video(
    TEST_VIDEO,
    OUTPUT_VIDEO,
    ALERT_CSV
)

alerts_df


requirements: Ultralytics requirement ['lap>=0.5.12'] not found, attempting AutoUpdate...
Using Python 3.13.16 environment at: /usr
Resolved 2 packages in 168ms
Prepared 1 package in 24ms
Installed 1 package in 1ms
 + lap==0.5.13

requirements: AutoUpdate success ✅ 0.5s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect

video1.mp4: 50/488
video1.mp4: 100/488
video1.mp4: 150/488
video1.mp4: 200/488
video1.mp4: 250/488
video1.mp4: 300/488
video1.mp4: 350/488
video1.mp4: 400/488
video1.mp4: 450/488

Finished: video1.mp4
Output video: /content/drive/MyDrive/cheating tracking/outputs/final_cheating/video1_final_cheating.mp4
Alert CSV: /content/drive/MyDrive/cheating tracking/outputs/final_cheating/alerts/video1_alerts.csv
Number of alerts: 0


""


## 18. Preview final processed video

In [18]:
display(
    Video(
        str(OUTPUT_VIDEO),
        embed=True,
        width=900
    )
)


Output hidden; open in https://colab.research.google.com to view.

## 19. Review alert history

In [19]:
if len(alerts_df) == 0:
    print("No cheating alerts were created.")
else:
    display(
        alerts_df[
            [
                "student_track_id",
                "behavior_detected",
                "confidence_score",
                "video_time_seconds",
                "event_datetime",
                "screenshot_path",
            ]
        ]
    )


No cheating alerts were created.


## 20. Show saved screenshots

In [20]:
import matplotlib.pyplot as plt

if len(alerts_df) > 0:

    for _, row in alerts_df.head(10).iterrows():

        image = cv2.imread(
            row["screenshot_path"]
        )

        if image is None:
            continue

        image_rgb = cv2.cvtColor(
            image,
            cv2.COLOR_BGR2RGB
        )

        plt.figure(
            figsize=(12, 7)
        )

        plt.imshow(
            image_rgb
        )

        plt.title(
            f"Student {row['student_track_id']} | "
            f"{row['behavior_detected']} | "
            f"{row['video_time_seconds']} sec"
        )

        plt.axis("off")
        plt.show()


## 21. Optional — Process all videos

Run after verifying the first final output.


In [21]:
all_alerts = []

for video_path in video_files:

    output_video = (
        FINAL_DIR /
        f"{video_path.stem}_final_cheating.mp4"
    )

    alert_csv = (
        ALERT_DIR /
        f"{video_path.stem}_alerts.csv"
    )

    df = process_final_cheating_video(
        video_path,
        output_video,
        alert_csv
    )

    if len(df) > 0:
        all_alerts.append(df)

if len(all_alerts) > 0:

    all_alerts_df = pd.concat(
        all_alerts,
        ignore_index=True
    )

else:
    all_alerts_df = pd.DataFrame()

all_alerts_df


video1.mp4: 50/488
video1.mp4: 100/488
video1.mp4: 150/488
video1.mp4: 200/488
video1.mp4: 250/488
video1.mp4: 300/488
video1.mp4: 350/488
video1.mp4: 400/488
video1.mp4: 450/488

Finished: video1.mp4
Output video: /content/drive/MyDrive/cheating tracking/outputs/final_cheating/video1_final_cheating.mp4
Alert CSV: /content/drive/MyDrive/cheating tracking/outputs/final_cheating/alerts/video1_alerts.csv
Number of alerts: 0
video2.mp4: 50/261
video2.mp4: 100/261
video2.mp4: 150/261
ALERT | video2.mp4 | Student 3 | Repeated head turns | 6.26s
ALERT | video2.mp4 | Student 10 | Prolonged sideways looking | 7.51s
video2.mp4: 200/261
ALERT | video2.mp4 | Student 1 | Prolonged sideways looking | 10.39s
video2.mp4: 250/261

Finished: video2.mp4
Output video: /content/drive/MyDrive/cheating tracking/outputs/final_cheating/video2_final_cheating.mp4
Alert CSV: /content/drive/MyDrive/cheating tracking/outputs/final_cheating/alerts/video2_alerts.csv
Number of alerts: 3
ALERT | video3.mp4 | Student 2 

,video,student_track_id,behavior_detected,confidence_score,video_time_seconds,event_datetime,screenshot_path,head_direction,side_look_duration,head_turn_count,phone_duration
0,video2.mp4,3,Repeated head turns,0.7500,6.256,2026-10-28 08:03:00,/content/drive/MyDrive/cheating tracking/outpu...,left,0.125,3,0.000
1,video2.mp4,10,Prolonged sideways looking,0.7001,7.507,2026-10-28 08:03:01,/content/drive/MyDrive/cheating tracking/outpu...,left,2.502,0,0.000
2,video2.mp4,1,Prolonged sideways looking,0.7001,10.385,2026-10-28 08:03:03,/content/drive/MyDrive/cheating tracking/outpu...,left,2.503,0,0.000
3,video3.mp4,2,Phone detected,0.8000,0.876,2026-10-28 08:03:05,/content/drive/MyDrive/cheating tracking/outpu...,forward,0.000,0,0.501
4,video3.mp4,4,Phone detected,0.8000,0.876,2026-10-28 08:03:05,/content/drive/MyDrive/cheating tracking/outpu...,unknown,0.000,0,0.501
5,video3.mp4,3,Prolonged sideways looking,0.7001,3.754,2026-10-28 08:03:06,/content/drive/MyDrive/cheating tracking/outpu...,left,2.503,0,0.000
6,video3.mp4,8,Phone detected,0.8000,4.338,2026-10-28 08:03:07,/content/drive/MyDrive/cheating tracking/outpu...,unknown,0.000,0,0.501
7,video3.mp4,2,Phone detected,0.8000,7.633,2026-10-28 08:03:08,/content/drive/MyDrive/cheating tracking/outpu...,forward,0.000,0,0.501
8,video3.mp4,4,Phone detected,0.8000,7.799,2026-10-28 08:03:08,/content/drive/MyDrive/cheating tracking/outpu...,unknown,0.000,0,0.667
9,video4.mp4,2,Prolonged sideways looking,0.7001,2.502,2026-10-28 08:03:13,/content/drive/MyDrive/cheating tracking/outpu...,left,2.502,0,0.000


## 22. Save one combined Alert History file

In [22]:
COMBINED_ALERT_HISTORY = (
    ALERT_DIR /
    "all_cheating_alerts.csv"
)

all_alerts_df.to_csv(
    COMBINED_ALERT_HISTORY,
    index=False
)

print(
    "Saved:",
    COMBINED_ALERT_HISTORY
)


Saved: /content/drive/MyDrive/cheating tracking/outputs/final_cheating/alerts/all_cheating_alerts.csv
